# PDV vs CD Comparison

Structure-adherence (SA) and task-accomplishment (TA) by task type for every model in `results/pdv_bt_all100`, paired with the same model's CD run from `results/bt_cd_all100_parallel`. Models that appear only under `bt_cd_all100_parallel` are excluded.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from statistics import mean

import pandas as pd
from IPython.display import display


# ---------- Paths ----------
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "tasks" / "tasks_100.json").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory.")


ROOT = find_repo_root(Path.cwd().resolve())
RESULTS_DIR = ROOT / "results"
TASKS_PATH = ROOT / "src" / "tasks" / "tasks_100.json"

PVD_ROOT = RESULTS_DIR / "pdv_bt_all100"
CD_ROOT = RESULTS_DIR / "bt_cd_all100_parallel"

RUN_ROOTS = {
    "CD": CD_ROOT,
    "PVD (Ours)": PVD_ROOT,
}

TASK_TYPE_ORDER = [
    ("face_target", "Face(Easy)"),
    ("go_to_target", "GoTo(Easy)"),
    ("go_around_obstacle", "Around(Med)"),
    ("move_to_closest_target", "Closest(Hard)"),
    ("go_to_multiple_targets", "MultiGoTo(Hard)"),
]

PREFERRED_MODEL_ORDER = [
    "Qwen2.5-3B-Instruct",
    "Qwen2.5-7B-Instruct",
    "Qwen2.5-14B-Instruct",
    "Qwen3-8B",
    "Qwen3-14B",
    "Llama-3.1-8B-Instruct",
    "phi-4",
    "granite-3.3-8b-instruct",
    "gemma-2-9b-it",
    "DeepSeek-R1-Distill-Qwen-14B",
]


def load_tasks_index(tasks_path: Path) -> tuple[dict[str, str], dict[str, int]]:
    tasks = json.loads(tasks_path.read_text(encoding="utf-8"))
    task_id_to_type: dict[str, str] = {}
    expected_per_type = {task_type: 0 for task_type, _ in TASK_TYPE_ORDER}

    for task in tasks:
        task_id = str(task["task_id"])
        task_type = task["task_type"]
        task_id_to_type[task_id] = task_type
        if task_type in expected_per_type:
            expected_per_type[task_type] += 1

    return task_id_to_type, expected_per_type


def clean_model_name(model_id: str) -> str:
    return model_id.split("/", 1)[1] if "/" in model_id else model_id


def collect_pvd_model_dirs() -> list[str]:
    if not PVD_ROOT.exists():
        raise FileNotFoundError(f"Missing PVD results directory: {PVD_ROOT}")

    return sorted(
        child.name
        for child in PVD_ROOT.iterdir()
        if child.is_dir() and not child.name.startswith(".")
    )


def load_mode_metrics(
    run_root: Path,
    model_dir: str,
    task_id_to_type: dict[str, str],
    expected_per_type: dict[str, int],
) -> dict[str, tuple[float | None, float | None]] | None:
    model_root = run_root / model_dir
    main_results_path = model_root / "main_results.json"
    tasks_dir = model_root / "tasks"

    if not main_results_path.exists() and not tasks_dir.exists():
        return None

    main_results = {}
    if main_results_path.exists():
        main_results = json.loads(main_results_path.read_text(encoding="utf-8"))

    metrics: dict[str, tuple[float | None, float | None]] = {
        task_type: (None, None) for task_type, _ in TASK_TYPE_ORDER
    }

    if not tasks_dir.exists():
        completion_map = main_results.get("completion_pct_by_task_type", {})
        for task_type, _ in TASK_TYPE_ORDER:
            ta = completion_map.get(f"{task_type}_pct")
            metrics[task_type] = (None, float(ta) if ta is not None else None)
        return metrics

    sums = {
        task_type: {
            "total_bt": 0,
            "valid_structure": 0,
            "completed": 0,
            "seen": 0,
        }
        for task_type, _ in TASK_TYPE_ORDER
    }

    for task_file in tasks_dir.glob("task_*.json"):
        task_id = task_file.stem.removeprefix("task_")
        task_type = task_id_to_type.get(task_id)
        if task_type not in sums:
            continue

        task_result = json.loads(task_file.read_text(encoding="utf-8"))
        sums[task_type]["seen"] += 1
        sums[task_type]["total_bt"] += int(task_result.get("bt_count", 0))
        sums[task_type]["valid_structure"] += int(task_result.get("valid_structure_count", 0))
        sums[task_type]["completed"] += 1 if bool(task_result.get("task_completion", False)) else 0

    for task_type, _ in TASK_TYPE_ORDER:
        expected = expected_per_type[task_type]
        seen = sums[task_type]["seen"]

        if seen < expected or expected == 0:
            metrics[task_type] = (None, None)
            continue

        total_bt = sums[task_type]["total_bt"]
        valid_structure = sums[task_type]["valid_structure"]
        completed = sums[task_type]["completed"]

        sa = (valid_structure / total_bt) * 100 if total_bt > 0 else None
        ta = (completed / expected) * 100 if expected > 0 else None
        metrics[task_type] = (sa, ta)

    return metrics


def fmt(value: float | None) -> str:
    if value is None or pd.isna(value):
        return "X"
    return f"{float(value):.1f}"


def resolve_display_name(model_dir: str) -> str:
    for run_root in (PVD_ROOT, CD_ROOT):
        main_results_path = run_root / model_dir / "main_results.json"
        if main_results_path.exists():
            main_results = json.loads(main_results_path.read_text(encoding="utf-8"))
            return clean_model_name(str(main_results.get("model_id", model_dir)))
    return model_dir


def order_model_dirs(model_dirs: list[str]) -> list[str]:
    display_to_dir = {resolve_display_name(model_dir): model_dir for model_dir in model_dirs}

    ordered: list[str] = []
    for preferred_name in PREFERRED_MODEL_ORDER:
        if preferred_name in display_to_dir:
            ordered.append(display_to_dir[preferred_name])

    remaining = sorted(set(model_dirs) - set(ordered))
    return ordered + remaining


# ---------- Build table ----------
task_id_to_type, expected_per_type = load_tasks_index(TASKS_PATH)
pvd_model_dirs = order_model_dirs(collect_pvd_model_dirs())

records = []
for model_dir in pvd_model_dirs:
    model_name = resolve_display_name(model_dir)

    for alg_label in ("CD", "PVD (Ours)"):
        mode_metrics = load_mode_metrics(
            RUN_ROOTS[alg_label],
            model_dir,
            task_id_to_type,
            expected_per_type,
        )

        row = {"Model": model_name, "Alg.": alg_label}
        sa_values: list[float] = []
        ta_values: list[float] = []

        for task_type, task_label in TASK_TYPE_ORDER:
            if mode_metrics is None:
                sa, ta = None, None
            else:
                sa, ta = mode_metrics[task_type]

            row[f"{task_label} SA"] = sa
            row[f"{task_label} TA"] = ta

            if sa is not None:
                sa_values.append(sa)
            if ta is not None:
                ta_values.append(ta)

        row["Avg. SA"] = mean(sa_values) if len(sa_values) == len(TASK_TYPE_ORDER) else None
        row["Avg. TA"] = mean(ta_values) if len(ta_values) == len(TASK_TYPE_ORDER) else None
        records.append(row)

df = pd.DataFrame(records)

ordered_pairs = [
    ("Face(Easy)", "SA"), ("Face(Easy)", "TA"),
    ("GoTo(Easy)", "SA"), ("GoTo(Easy)", "TA"),
    ("Around(Med)", "SA"), ("Around(Med)", "TA"),
    ("Closest(Hard)", "SA"), ("Closest(Hard)", "TA"),
    ("MultiGoTo(Hard)", "SA"), ("MultiGoTo(Hard)", "TA"),
    ("Avg.", "SA"), ("Avg.", "TA"),
]
flat_to_grouped = {
    "Face(Easy) SA": ("Face(Easy)", "SA"), "Face(Easy) TA": ("Face(Easy)", "TA"),
    "GoTo(Easy) SA": ("GoTo(Easy)", "SA"), "GoTo(Easy) TA": ("GoTo(Easy)", "TA"),
    "Around(Med) SA": ("Around(Med)", "SA"), "Around(Med) TA": ("Around(Med)", "TA"),
    "Closest(Hard) SA": ("Closest(Hard)", "SA"), "Closest(Hard) TA": ("Closest(Hard)", "TA"),
    "MultiGoTo(Hard) SA": ("MultiGoTo(Hard)", "SA"), "MultiGoTo(Hard) TA": ("MultiGoTo(Hard)", "TA"),
    "Avg. SA": ("Avg.", "SA"), "Avg. TA": ("Avg.", "TA"),
}

actual_table = df.set_index(["Model", "Alg."]).rename(columns=flat_to_grouped)
actual_table = actual_table[[flat_to_grouped[k] for k in flat_to_grouped]]
actual_table.columns = pd.MultiIndex.from_tuples(actual_table.columns)
actual_table = actual_table.reindex(columns=pd.MultiIndex.from_tuples(ordered_pairs))
formatted_table = actual_table.apply(lambda col: col.map(fmt))


def style_grouped_table(table: pd.DataFrame) -> pd.io.formats.style.Styler:
    model_names = list(table.index.get_level_values("Model").unique())

    def row_style(row: pd.Series) -> list[str]:
        model = row.name[0]
        alg = row.name[1]
        if alg == "CD" and model != model_names[0]:
            return ["border-top: 2px solid #444"] * len(row)
        return [""] * len(row)

    return table.style.apply(row_style, axis=1)


display(style_grouped_table(formatted_table))
print(f"Models from {PVD_ROOT.name}: {len(pvd_model_dirs)}")
print("Included model dirs:", ", ".join(pvd_model_dirs))

Models from pdv_bt_all100: 10
Included model dirs: Qwen2.5-3B-Instruct, Qwen2.5-7B-Instruct, Qwen2.5-14B-Instruct, Qwen3-8B, Qwen3-14B, Llama-3.1-8B-Instruct, phi-4, granite-3.3-8b-instruct, gemma-2-9b-it, DeepSeek-R1-Distill-Qwen-14B


In [ ]:
# ---------- LaTeX export (paper table) ----------

def build_latex_table(table_df: pd.DataFrame) -> str:
    lines: list[str] = []

    lines.append(r"\begin{tabular}{@{}l l cc cc cc cc cc cc@{}}")
    lines.append(r"\toprule")
    lines.append(
        r"& "
        + r"& \multicolumn{2}{c}{\textbf{Face}$^{E}$}"
        + r"& \multicolumn{2}{c}{\textbf{GoTo}$^{E}$}"
        + r"& \multicolumn{2}{c}{\textbf{Around}$^{M}$}"
        + r"& \multicolumn{2}{c}{\textbf{Closest}$^{H}$}"
        + r"& \multicolumn{2}{c}{\textbf{MultiGoTo}$^{H}$}"
        + r"& \multicolumn{2}{c}{\textbf{Avg.}} \\"
    )
    lines.append(
        r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}\cmidrule(lr){7-8}"
        r"\cmidrule(lr){9-10}\cmidrule(lr){11-12}\cmidrule(lr){13-14}"
    )
    lines.append(
        r"\textbf{Model} & \textbf{Alg.}"
        + r"& SA & TA & SA & TA & SA & TA & SA & TA & SA & TA & SA & TA \\"
    )
    lines.append(r"\midrule")

    grouped_models = list(table_df["Model"].drop_duplicates())
    for model_index, model_name in enumerate(grouped_models):
        model_rows = table_df[table_df["Model"] == model_name]

        row_cd = model_rows[model_rows["Alg."] == "CD"].iloc[0] if (model_rows["Alg."] == "CD").any() else None
        row_pvd = model_rows[model_rows["Alg."] == "PVD (Ours)"].iloc[0] if (model_rows["Alg."] == "PVD (Ours)").any() else None

        def render_cells(row) -> list[str]:
            if row is None:
                return ["X"] * 12
            out: list[str] = []
            for _, label in TASK_TYPE_ORDER:
                out.append(fmt(row[f"{label} SA"]))
                out.append(fmt(row[f"{label} TA"]))
            out.append(fmt(row["Avg. SA"]))
            out.append(fmt(row["Avg. TA"]))
            return out

        cd_cells = " & ".join(render_cells(row_cd))
        pvd_cells = " & ".join(render_cells(row_pvd))

        lines.append(rf"\multirow{{2}}{{*}}{{{model_name}}}")
        lines.append(rf"& CD & {cd_cells} \\")
        lines.append(rf"& PVD (Ours) & {pvd_cells} \\")

        if model_index < len(grouped_models) - 1:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    return "\n".join(lines)


latex_table = build_latex_table(df)
print(latex_table)